# ANNIE CC Studies

In [ ]:
pythonuserbase = "/exp/annie/app/users/mastbaum/env1/.local"
import sys
pyver = f"python{sys.version_info.major}.{sys.version_info.minor}"
sys.path.insert(0, f"{pythonuserbase}/lib/{pyver}/site-packages")

In [ ]:
import numpy as np
import uproot
import awkward as ak
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import matplotlib.lines as lines
from astropy.stats import binom_conf_interval
from matplotlib.colors import LogNorm
import scipy
import pandas as pd

In [ ]:
M_MU = 105.6583755  # MeV/c^2

# Event categories from EventCategory.hh enum EventCategory
categories = [
    'Unknown',
    'Signal CCQE', 'Signal CCMEC', 'Signal CCRES', 'Signal Other',
    r'$\nu_\mu$ CC$N\pi$', r'$\nu_\mu$ CC$0\pi 0p$', r'$\nu_\mu$ CC Other',
    r'$\nu_e$ CC', 'NC', 'OOFV', 'Other',
]

category_colors = np.array([
    (204, 204, 204),
    (0, 255, 0), (0, 204, 0), (0, 153, 0), (0, 102, 0),
    (51, 102, 255), (0, 51, 204), (0, 51, 255),
    (204, 0, 255), (255, 204, 0), (102, 0, 0), (204, 0, 0)
]) / 255

### Tree reader classes

To normalize reco variables across truth, simpleReco, cMRD

In [ ]:
class NormTree(object):
    '''A container to normalize the fields of various recos.'''
    LABEL = None
    FIELD_MAP = None
    def __init__(self, tree, label=LABEL):
        self.label = label
        self.tree = tree
        self.field_cache = {
            k: tree[v].array() for k, v in self.FIELD_MAP.items()
        }

        e = tree['trueMuonEnergy'].array()
        self.field_cache['pmu_true'] = np.sqrt(np.square(e) - 105.7**2)

        self.field_cache['true_cc'] = (self.trueCC == 1) & (self.trueNuPDG  == 14)

    def _build_flags(self):
        self.field_cache['is_forward'] = self.ctz > 0.8
        self.field_cache['is_pmu_range'] = (self.pmu > 600) & (self.pmu < 1200)        
    
    def __getattr__(self, k):
        if k in self.field_cache.keys():
            return self.field_cache[k]

        v = self.tree[k].array()
        self.field_cache[k] = v
        return self.field_cache[k]

    def __getitem__(self, k):
        return self.__getattr__(k)

    def plot_distribution(self, var, bins, color, axis, sel_mask=None):
        if sel_mask is None: sel_mask = self.signal
        v = self[var]
        axis.hist(v[sel_mask], histtype='step', bins=bins, color=color, label=self.LABEL)

    def plot_smearing(self, var_true, var_reco, bins, axis, sel_mask=None):
        if sel_mask is None: sel_mask = self.signal
        vt, vr = np.array(self[var_true]), np.array(self[var_reco])
        axis.hist2d(vt[sel_mask], vr[sel_mask], bins=bins)
        axis.axline((0,0), (1,1), color='red', ls='--')
        axis.set_xlim(bins[0], bins[-1])
        axis.set_ylim(bins[0], bins[-1])
        axis.set_xlabel(var_true)
        axis.set_ylabel(var_reco)

    def plot_residual(self, var_true, var_reco, bins, color, axis, sel_mask=None, stack=True, label=None):
        if sel_mask is None: sel_mask = self.signal
        vt, vr = np.array(self[var_true])[sel_mask], np.array(self[var_reco])[sel_mask]
        residual = (vr - vt) / vt
        if label is None: label = self.LABEL
        axis.hist(residual, histtype='step', bins=bins, color=color, label=label)

        if stack:
            # Stacked by true event category
            cats = [np.array(residual[self.category[sel_mask] == cat]) for cat in np.unique(self.category)][::-1]
            labels = [categories[cat] for cat in np.unique(self.category)][::-1]
            colors = [category_colors[cat] for cat in np.unique(self.category)][::-1]
            axis.hist(cats, histtype='barstacked', bins=bins, label=labels, color=colors)
        
        axis.grid(axis='x')
        axis.set_xlim(bins[0], bins[-1])
        axis.set_xlabel('(%s - %s) / %s' % (var_reco, var_true, var_true))
        axis.legend(loc='best', fontsize=8)
    
    def plot_efficiency(self, var, bins, color, axis, sig_mask=None, sel_mask=None):
        if sig_mask is None: sig_mask = self.mc_ccinc_signal
        if sel_mask is None: sel_mask = self.signal
        v = self[var]
            
        h_sig, _ = np.histogram(v[sig_mask], bins=bins)
        h_sel, _ = np.histogram(v[sel_mask], bins=bins)
        h_selsig, _ = np.histogram(v[sig_mask & sel_mask], bins=bins)

        x = bins[:-1] + np.diff(bins)[0]/2
        eff = h_selsig / h_sig
        e_eff = binom_conf_interval(h_selsig, h_sig)
        e_eff = eff - e_eff[0], e_eff[1] - eff
        axis.errorbar(x, eff, e_eff, np.diff(bins)/2, fmt='.', ms=10, color=color, label=self.LABEL)
    
    def plot_purity(self, var, bins, color, axis, sig_mask=None, sel_mask=None):
        if sig_mask is None: sig_mask = self.mc_ccinc_signal
        if sel_mask is None: sel_mask = self.signal
        v = self[var]
            
        h_sig, _ = np.histogram(v[sig_mask], bins=bins)
        h_sel, _ = np.histogram(v[sel_mask], bins=bins)
        h_selsig, _ = np.histogram(v[sig_mask & sel_mask], bins=bins)
        
        x = bins[:-1] + np.diff(bins)[0]/2
        pur = h_selsig / h_sel
        e_pur = binom_conf_interval(h_selsig, h_sel)
        e_pur = pur - e_pur[0], e_pur[1] - pur
        axis.errorbar(x, pur, e_pur, np.diff(bins)/2, fmt='.', ms=10, color=color, label=self.LABEL)

class Truth(NormTree):
    LABEL = 'Truth'
    FIELD_MAP = {
        'signal': 'mc_ccinc_signal',
        'in_fv': 'trueFV',
    }

    def __init__(self, tree):
        super().__init__(tree)

        self.field_cache['vx'] = tree['trueNuIntxVtx_X'].array()
        self.field_cache['vy'] = tree['trueNuIntxVtx_Y'].array() + 14.46
        self.field_cache['vz'] = tree['trueNuIntxVtx_Z'].array() - 168.1
        self.field_cache['ctz'] = np.cos(tree['trueAngle'].array() * 180/np.pi)
        
        e = tree['trueMuonEnergy'].array()
        self.field_cache['pmu'] = np.sqrt(np.square(e) - 105.7**2)

        self.field_cache['valid'] = np.ones_like(self.signal)
        self.field_cache['presel'] = self.true_cc

        self._build_flags()

class SimpleReco(NormTree):  
    LABEL = 'simpleReco'
    FIELD_MAP = {
        'signal': 'sel_ccinc_simple',
        'in_fv': 'recoFV',
        'ctz': 'simpleRecoCosTheta',
        'pmu_corr': 'simpleRecoMomentumCor',
    }

    def __init__(self, tree):
        super().__init__(tree)

        self.field_cache['vx'] = tree['simpleRecoVtxX'].array() * 100
        self.field_cache['vy'] = tree['simpleRecoVtxY'].array() * 100 + 14.46
        self.field_cache['vz'] = tree['simpleRecoVtxZ'].array() * 100 - 168.1

        e = tree['simpleRecoEnergy'].array() + M_MU
        self.field_cache['pmu'] = np.sqrt(np.square(e) - M_MU**2)

        flag = tree['simpleRecoFlag'].array()
        self.field_cache['valid'] = (flag == 1) & ~np.isnan(self.vx) & (self.vx > -9998)
        self.field_cache['presel'] = (
              (tree['trigword'].array() == 5)
            & (tree['HasTank'].array() == 1)
            & (tree['HasMRD'].array() == 1)
            & (tree['TankMRDCoinc'].array() == 1)
            & (tree['NoVeto'].array() == 1)
            & (tree['recoInc_contained_in_MRD'].array() == 1)
            & self.valid
        )

        self._build_flags()

class CMRDReco(NormTree):  
    LABEL = 'cMRD'
    FIELD_MAP = {
        'signal': 'sel_ccinc_cmrd',
        'in_fv': 'recoFV_cmrd',
        'ctz': 'mrd_ml_cos_theta',
    }

    def __init__(self, tree):
        super().__init__(tree)

        vtx = tree['mrd_ml_tank_vertex_cm'].array()
        self.field_cache['vx'] = vtx[:,0]
        self.field_cache['vy'] = vtx[:,1] + 14.46
        self.field_cache['vz'] = vtx[:,2] - 168.1
        
        e = tree['mrd_ml_muon_ke_mev'].array() + M_MU
        self.field_cache['pmu'] = np.sqrt(np.square(e) - M_MU**2)

        self.field_cache['valid'] = (
              tree["mrd_ml_reconstruction_available"].array()
            & tree["mrd_ml_reco_structurally_eligible"].array()
            & (tree["mrd_ml_topology_class"].array() == 0)
            & (tree["mrd_ml_input_tank_hits"].array()  > 0)
        )
        self.field_cache['presel'] = (
              (tree['trigword'].array() == 5)
            & (tree['HasTank'].array() == 1)
            & (tree['HasMRD'].array() == 1)
            & (tree['TankMRDCoinc'].array() == 1)
            & (tree['NoVeto'].array() == 1)
            & self.valid
        )

        self._build_flags()

## Load STV Trees

In [ ]:
f = uproot.open('/exp/annie/data/users/mastbaum/cc/mc/PhaseIITree_merged.stv.root')
t = f['phaseIITriggerTree']
t.num_entries

In [ ]:
truth = Truth(t)
reco_sr = SimpleReco(t)
reco_cm = CMRDReco(t)

List branches

In [ ]:
t.show()

## Reconstruction and Units

### Position

Vertex positions have a known offset to place the tank center at the origin:

* $x \to x$
* $y \to y + 14.46$ cm
* $z \to z - 168.1$ cm

Note that truth and cMRD vertices are in cm, while simpleReco is in meters.

### Energy and Momentum

Energies are MeV, momenta MeV/$c$, masses MeV/$c^2$.

Truth `trueMuonEnergy` is the muon *total* energy. 

In [ ]:
np.min(truth.trueMuonEnergy[truth.presel])

`simpleRecoEnergy` is reconstructed muon *kinetic* energy in MeV. We can show this using `simpleRecoPt` and `simpleRecoCosTheta`.

In [ ]:
m = reco_sr.valid

e_sr = reco_sr.simpleRecoEnergy[m]
p_sr = reco_sr.simpleRecoPt[m] / np.sin(np.acos(reco_sr.simpleRecoCosTheta[m]))
etot_sr = np.sqrt(np.square(p_sr) + M_MU**2)
ekin_sr = etot_sr - M_MU

np.nanmean(etot_sr - e_sr), np.nanmean(ekin_sr - e_sr)

#### simpleReco Tuning

`simpleReco` KE does not quite align with true muon KE, when looking at all valid fits.

In [ ]:
ekin_true = reco_sr.trueMuonEnergy[m] - M_MU
np.nanmean(e_sr - ekin_true)

Pre-selection:

In [ ]:
np.nanmean(reco_sr.simpleRecoEnergy[reco_sr.presel] - (reco_sr.trueMuonEnergy[reco_sr.presel] - M_MU))

Selection:

In [ ]:
np.nanmean(reco_sr.simpleRecoEnergy[reco_sr.signal] - (reco_sr.trueMuonEnergy[reco_sr.signal] - M_MU))

There is a large tail of OOFV events reconstructing with too-high energies:

In [ ]:
reco = reco_sr
m = reco.signal & reco.mc_ccinc_signal
var_true, var_reco = 'pmu_true', 'pmu'

fig, axes = plt.subplots(1, 2, figsize=(8, 4), tight_layout=True)
reco.plot_smearing(var_true, var_reco, np.linspace(200, 1600, 51), axes[0], sel_mask=m)
reco.plot_residual(var_true, var_reco, np.linspace(-1, 1, 51), 'black', axes[1], sel_mask=m, label='')
reco.plot_residual(var_true, var_reco, np.linspace(-1, 1, 51), 'gray', axes[1], sel_mask=reco_sr.signal, stack=False, label='Selected')
axes[1].grid(axis='x')
plt.show()

print(np.nanmean(reco_sr[var_reco][m] - (reco_sr[var_true][m])))

## Distributions

In [ ]:
var = 'pmu'
label = r'$p_\mu$ [MeV]'
bins = np.linspace(0, 1800, 51)

fig, ax = plt.subplots(figsize=(4, 4), tight_layout=True)

for i, reco in enumerate((truth, reco_sr, reco_cm)):
    reco.plot_distribution(var, bins, 'C%i' % i, ax, sel_mask=reco.presel&reco.in_fv)

ax.set_xlim(bins[0], bins[-1])
ax.legend(loc='best')
ax.set_xlabel(label)
ax.set_ylabel('Events')
plt.show()

## Residuals

In [ ]:
var_true, var_reco = 'pmu_true', 'pmu'

fig, axes = plt.subplots(2, 2, figsize=(8, 8), tight_layout=True)

for i, reco in enumerate((reco_sr, reco_cm)):
    axes[i,0].set_title(reco.LABEL)
    m = reco.presel & reco.in_fv & (reco.recoPE == 1)
    reco.plot_smearing(var_true, var_reco, np.linspace(400, 1400, 51), axes[i,0], sel_mask=m)
    reco.plot_residual(var_true, var_reco, np.linspace(-1, 1, 51), 'black', axes[i,1], sel_mask=m)

plt.show()

## Efficiency and Purity

At pre-selection:

In [ ]:
var = 'pmu_true'
label = r'$p_\mu^\mathrm{true}$ [MeV]'
bins = np.linspace(0, 1800, 16)

fig, axes = plt.subplots(1, 2, figsize=(8, 4), tight_layout=True)

for i, reco in enumerate((reco_sr, reco_cm)):
    sig = reco.true_cc & reco.is_forward
    sel = reco.presel
    reco.plot_efficiency(var, bins, 'C%i' % i, axes[0], sig_mask=sig, sel_mask=sel)
    reco.plot_purity(var, bins, 'C%i' % i, axes[1], sig_mask=sig, sel_mask=sel)

for ax in axes:
    ax.set_xlim(bins[0], bins[-1])
    ax.set_ylim(0, 1)
    ax.legend(loc='lower right')
    ax.set_xlabel(label)
    ax.grid(axis='y')

axes[0].set_ylabel('Efficiency')
axes[1].set_ylabel('Purity')
plt.show()

## Vertex

In [ ]:
bins = np.linspace(-300, 300, 101)
hkw = {'bins': bins, 'cmap': 'coolwarm'}
pkw = {'linewidth': 1, 'ls': '--', 'edgecolor': 'r', 'facecolor': 'none'}
pkw2 = {'linewidth': 1, 'ls': '--', 'edgecolor': 'black', 'facecolor': 'none'}

for i, model in enumerate((truth, reco_sr, reco_cm)):
    fig, ax = plt.subplots(1, 3, figsize=(9, 3), tight_layout=True)

    m = model.presel
    ax[0].hist2d(np.array(model.vx[m]), np.array(model.vy[m]), **hkw)
    ax[0].set_xlabel('%s vertex $x$ [cm]' % model.LABEL)
    ax[0].set_ylabel('%s vertex $y$ [cm]' % model.LABEL)
    fv0 = patches.Rectangle((-100, -100), 200, 200, **pkw)
    ax[0].add_patch(fv0)
    t0 = patches.Rectangle((-150, -200), 300, 400, **pkw2)
    ax[0].add_patch(t0)
    
    ax[1].hist2d(np.array(model.vz[m]), np.array(model.vy[m]), **hkw)
    ax[1].set_xlabel('%s vertex $z$ [cm]' % model.LABEL)
    ax[1].set_ylabel('%s vertex $y$ [cm]' % model.LABEL)
    fv1 = patches.Rectangle((-100, -100), 200, 200, **pkw)
    ax[1].add_patch(fv1)
    t1 = patches.Rectangle((-150, -200), 300, 400, **pkw2)
    ax[1].add_patch(t1)
    
    ax[2].hist2d(np.array(model.vx[m]), np.array(model.vz[m]), **hkw)
    ax[2].set_xlabel('%s vertex $x$ [cm]' % model.LABEL)
    ax[2].set_ylabel('%s vertex $z$ [cm]' % model.LABEL)
    fv2 = patches.Circle((0, 0), 100, **pkw)
    ax[2].add_patch(fv2)
    t2 = patches.Circle((0, 0), 150, **pkw2)
    ax[2].add_patch(t2)
